In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# 4 classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# If in Colab and you need to upload:
# from google.colab import files; files.upload()

df = pd.read_csv("diabetes.csv")

# Replace medically-impossible zeros -> NaN -> median
zero_cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
for c in zero_cols:
    df.loc[df[c] == 0, c] = np.nan
    df[c] = df[c].fillna(df[c].median())

# Optional small discretization feature from BMI
bmi_bins = [0, 18.5, 25, 30, np.inf]
df["BMI_Bin"] = pd.cut(df["BMI"], bins=bmi_bins,
                       labels=["under","normal","over","obese"], right=False
                      ).astype("category").cat.codes

y = df["Outcome"].astype(int)
X = df.drop(columns=["Outcome"])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


In [ ]:
models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "NaiveBayes": GaussianNB(),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

def eval_model(name, clf):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    proba = (clf.predict_proba(X_test)[:,1] if hasattr(clf,"predict_proba")
             else clf.decision_function(X_test) if hasattr(clf,"decision_function")
             else None)

    acc = accuracy_score(y_test, preds)
    pre = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1  = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, proba) if proba is not None else np.nan
    cm  = confusion_matrix(y_test, preds)

    print(f"\n=== {name} ===")
    print(f"Acc:{acc:.3f}  Prec:{pre:.3f}  Rec:{rec:.3f}  F1:{f1:.3f}  "
          + (f"AUC:{auc:.3f}" if not np.isnan(auc) else "AUC:N/A"))
    print("Confusion Matrix:\n", cm)

for name, clf in models.items():
    eval_model(name, clf)



=== LogReg ===
Acc:0.736  Prec:0.656  Rec:0.519  F1:0.579  AUC:0.839
Confusion Matrix:
 [[128  22]
 [ 39  42]]

=== DecisionTree ===
Acc:0.697  Prec:0.568  Rec:0.568  F1:0.568  AUC:0.667
Confusion Matrix:
 [[115  35]
 [ 35  46]]

=== NaiveBayes ===
Acc:0.723  Prec:0.602  Rec:0.617  F1:0.610  AUC:0.802
Confusion Matrix:
 [[117  33]
 [ 31  50]]

=== SVM ===
Acc:0.745  Prec:0.677  Rec:0.519  F1:0.587  AUC:0.815
Confusion Matrix:
 [[130  20]
 [ 39  42]]


In [ ]:
rows = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    proba = (clf.predict_proba(X_test)[:,1] if hasattr(clf,"predict_proba")
             else clf.decision_function(X_test) if hasattr(clf,"decision_function")
             else None)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, proba) if proba is not None else np.nan
    })

pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,NaiveBayes,0.722944,0.602410,0.617284,0.609756,0.802140
1,SVM,0.744589,0.677419,0.518519,0.587413,0.815473
2,LogReg,0.735931,0.656250,0.518519,0.579310,0.839424
3,DecisionTree,0.696970,0.567901,0.567901,0.567901,0.667284


In [ ]:
# === Example of external record (fill with your own values) ===
# Format: [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin,
#          BMI, DiabetesPedigreeFunction, Age, BMI_Bin]
# Note: BMI_Bin optional, we will recompute it below anyway

new_record = {
    "Pregnancies": 2,
    "Glucose": 130,
    "BloodPressure": 70,
    "SkinThickness": 25,
    "Insulin": 100,
    "BMI": 28.5,
    "DiabetesPedigreeFunction": 0.5,
    "Age": 35
}

import pandas as pd

# Put into DataFrame
new_df = pd.DataFrame([new_record])

# Handle zeros if any
for c in ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]:
    if new_df.loc[0, c] == 0:
        new_df[c] = df[c].median()  # use train median

# Add BMI_Bin same as training
bmi_bins = [0, 18.5, 25, 30, np.inf]
bmi_labels = ["under","normal","over","obese"]
new_df["BMI_Bin"] = pd.cut(
    new_df["BMI"], bins=bmi_bins, labels=bmi_labels, right=False
).astype("category").cat.codes

# Reorder features same as training
new_df = new_df[X.columns]

# Scale with the SAME scaler used in training
new_scaled = scaler.transform(new_df)

# Predict with all models
print("=== Predictions for external record ===")
for name, clf in models.items():
    pred = clf.predict(new_scaled)[0]
    label = "Diabetic" if pred==1 else "Non-Diabetic"
    print(f"{name}: {label}")


=== Predictions for external record ===
LogReg: Non-Diabetic
DecisionTree: Non-Diabetic
NaiveBayes: Non-Diabetic
SVM: Non-Diabetic
